# Voyage Analytics — Containerisation

**Notebook:** `10_Docker` · **Objective:** #3 (package and deploy with Docker)
**Files:** `Dockerfile` · `.dockerignore` · `docker-compose.yml` · `docker/requirements-serve.txt`

---

### What gets containerised
Only the **prediction API**. The training pipeline stays outside the image: a container that answers
HTTP has no reason to carry MLflow, Jupyter, matplotlib or the tuning stack.

```
voyage-analytics-api
├── src/serving/     Flask app
├── src/models/      model classes needed to unpickle the artifacts
├── src/features/    shared preprocessor
├── src/utils/       config + logging
├── models/*.joblib  the three fitted pipelines
└── models/route_reference.json
```

### Four decisions worth explaining

**1. Multi-stage build.** Dependencies compile in a `builder` stage with `gcc`/`g++`; only the
resulting `site-packages` tree is copied into the runtime image. Compilers never ship to production.

**2. A separate, pinned serving requirements file.** `docker/requirements-serve.txt` lists 7 packages
instead of the project's 20. Critically, `scikit-learn`, `numpy` and `scipy` are pinned to the exact
versions the models were **fitted** with — a joblib artifact carries no version metadata, so a
mismatch surfaces as a corrupt-looking unpickle or, worse, silently different predictions.

**3. `waitress`, not Flask's dev server.** The built-in server is single-threaded and explicitly not
for production traffic.

**4. A health check with a 40-second start period.** This one comes straight from a measurement:
loading the gradient-boosting pipeline cold takes **~9 seconds**. Without a generous
`start_period` an orchestrator would kill the container before it ever became ready.


## 0. Setup

In [1]:
import subprocess, json, shutil, time, sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
print("project root:", ROOT)

def sh(cmd, timeout=600):
    """Run a shell command, returning (exit_code, output)."""
    try:
        r = subprocess.run(cmd, shell=True, cwd=ROOT, capture_output=True,
                           text=True, timeout=timeout)
        return r.returncode, (r.stdout + r.stderr).strip()
    except subprocess.TimeoutExpired:
        return -1, f"timed out after {timeout}s"

def docker_available():
    if shutil.which("docker") is None:
        return False, "docker CLI not installed"
    rc, out = sh("docker info --format {{.ServerVersion}}", timeout=30)
    return (rc == 0), (out.splitlines()[-1] if out else "daemon not reachable")

DOCKER_OK, detail = docker_available()
print("docker daemon:", "available - " + detail if DOCKER_OK else "NOT running")
if not DOCKER_OK:
    print("\nThe build/run cells below will be skipped and show the commands instead.")
    print("Start Docker Desktop and re-run to execute them for real.")

project root: F:\Final Project Labmentix\voyage-analytics


docker daemon: NOT running

The build/run cells below will be skipped and show the commands instead.
Start Docker Desktop and re-run to execute them for real.


## 1. The image definition

In [2]:
print((ROOT / "Dockerfile").read_text(encoding="utf-8"))

# syntax=docker/dockerfile:1
#
# Voyage Analytics — prediction API image (project objective #3)
#
# Multi-stage: dependencies are compiled in a builder stage and only the
# resulting site-packages are copied into the runtime image, so build tools
# never ship to production.
#
#   docker build -t voyage-analytics-api .
#   docker run -p 5000:5000 voyage-analytics-api

# ----------------------------------------------------------------- builder ---
FROM python:3.12-slim AS builder

WORKDIR /build

# Build-only toolchain; discarded with this stage.
RUN apt-get update \
    && apt-get install -y --no-install-recommends gcc g++ \
    && rm -rf /var/lib/apt/lists/*

COPY docker/requirements-serve.txt .

# --prefix keeps everything in one tree that the runtime stage can copy wholesale.
RUN pip install --no-cache-dir --prefix=/install -r requirements-serve.txt


# ----------------------------------------------------------------- runtime ---
FROM python:3.12-slim AS runtime

LABEL org.opencontai

### What `.dockerignore` keeps out — and the one thing it must keep in

The build context excludes `data/`, `notebooks/`, `mlflow/`, `tests/` and `.venv/`. Without this the
context would include ~100 MB of CSVs the API never reads.

**The subtlety:** `models/` is in `.gitignore` (fitted artifacts don't belong in version control) but
must **not** be in `.dockerignore` — the models are the entire point of the image. Getting these two
files backwards produces an image that builds cleanly and then 500s on every request.

In [3]:
print((ROOT / ".dockerignore").read_text(encoding="utf-8"))

# Keep the build context small and the image free of anything the API cannot use.
#
# NOTE: models/ is deliberately NOT ignored — the fitted pipelines are the whole
# point of the image. It appears in .gitignore (artifacts don't belong in git)
# but must be present here.

# ---- data (the API never reads raw or intermediate data) ----
data/
reports/

# ---- notebooks & analysis ----
notebooks/
*.ipynb
.ipynb_checkpoints/

# ---- experiment tracking ----
mlflow/
mlruns/
models/tuned/

# ---- local environment ----
.venv/
venv/
env/
__pycache__/
*.py[cod]
*.egg-info/

# ---- vcs / editors / os ----
.git/
.gitignore
.github/
.vscode/
.idea/
.DS_Store
Thumbs.db

# ---- tests & docs (not needed at runtime) ----
tests/
docs/
*.md
!README.md

# ---- misc ----
*.log
.env
pipelines/



## 2. Compose stack

In [4]:
print((ROOT / "docker-compose.yml").read_text(encoding="utf-8"))

# Voyage Analytics — local stack
#
#   docker compose up --build            # API only
#   docker compose --profile tracking up # API + MLflow UI
#   docker compose down
#
# The API image is self-contained; MLflow is a separate optional service so the
# production image never carries the tracking stack.

services:
  api:
    build:
      context: .
      dockerfile: Dockerfile
    image: voyage-analytics-api:latest
    container_name: voyage-api
    ports:
      - "5000:5000"
    environment:
      PORT: 5000
    healthcheck:
      test: ["CMD", "python", "-c",
             "import urllib.request,sys; sys.exit(0 if urllib.request.urlopen('http://localhost:5000/health', timeout=4).status==200 else 1)"]
      interval: 30s
      timeout: 5s
      # Generous start period: the container warms every model before it is ready.
      start_period: 40s
      retries: 3
    restart: unless-stopped
    # A prediction service needs no write access to its own filesystem.
    read_only: true
    tmp

In [5]:
# Compose syntax can be validated without a running daemon
rc, out = sh("docker compose config --quiet")
print("compose syntax:", "VALID" if rc == 0 else f"INVALID\n{out}")
rc, out = sh("docker compose config --services")
print("services (default profile):", out)
print("note: 'mlflow' is hidden because it sits behind the 'tracking' profile")

compose syntax: VALID


services (default profile): api
note: 'mlflow' is hidden because it sits behind the 'tracking' profile


## 3. Hardening

| Setting | Why |
|---|---|
| `USER appuser` (uid 1000) | a prediction service has no reason to run as root |
| `read_only: true` | the container never writes to its own filesystem |
| `tmpfs: /tmp` | the one writable path it does need |
| `no-new-privileges` | blocks privilege escalation via setuid binaries |
| `cpus: 2.0`, `memory: 1G` | a runaway request cannot starve the host |

`read_only` had a consequence worth noting: the API used to rebuild
`models/route_reference.json` when it was missing, which would fail on a read-only filesystem. The
fallback now builds it **in memory** (`save=False`) — in the image the file is baked in anyway, so
that path is only for local development.

## 4. Build the image

In [6]:
if DOCKER_OK:
    t0 = time.time()
    rc, out = sh("docker build -t voyage-analytics-api:latest .", timeout=1800)
    print(f"exit {rc}  ({time.time()-t0:.0f}s)")
    print("\n".join(out.splitlines()[-25:]))
else:
    print("SKIPPED - daemon not running. Command:")
    print("  docker build -t voyage-analytics-api:latest .")

SKIPPED - daemon not running. Command:
  docker build -t voyage-analytics-api:latest .


In [7]:
if DOCKER_OK:
    rc, out = sh("docker images voyage-analytics-api --format "
                 "\"{{.Repository}}:{{.Tag}}  {{.Size}}  created {{.CreatedSince}}\"")
    print(out or "image not found")
else:
    print("SKIPPED - run `docker images voyage-analytics-api` once the daemon is up")

SKIPPED - run `docker images voyage-analytics-api` once the daemon is up


## 5. Run it and verify the endpoints

In [8]:
if DOCKER_OK:
    sh("docker rm -f voyage-api-test")                      # clean any previous run
    rc, out = sh("docker run -d --name voyage-api-test -p 5001:5000 "
                 "voyage-analytics-api:latest")
    print("started:", out[:12] if rc == 0 else out)

    # Wait for the health check to pass — the container warms every model first.
    import urllib.request
    for attempt in range(60):
        try:
            with urllib.request.urlopen("http://localhost:5001/health", timeout=3) as r:
                if r.status == 200:
                    print(f"healthy after ~{attempt+1}s")
                    break
        except Exception:
            time.sleep(1)
    else:
        print("did not become healthy in 60s")
        print(sh("docker logs voyage-api-test")[1][-1500:])
else:
    print("SKIPPED. Commands:")
    print("  docker run -d --name voyage-api -p 5000:5000 voyage-analytics-api:latest")
    print("  curl http://localhost:5000/health")

SKIPPED. Commands:
  docker run -d --name voyage-api -p 5000:5000 voyage-analytics-api:latest
  curl http://localhost:5000/health


In [9]:
if DOCKER_OK:
    import urllib.request
    def call(path, body=None):
        url = f"http://localhost:5001{path}"
        if body is None:
            req = urllib.request.Request(url)
        else:
            req = urllib.request.Request(
                url, data=json.dumps(body).encode(),
                headers={"Content-Type": "application/json"})
        with urllib.request.urlopen(req, timeout=10) as r:
            return json.loads(r.read())

    print("health :", call("/health")["status"])
    p = call("/predict/flight-price", {"from": "Sao Paulo (SP)", "to": "Rio de Janeiro (RJ)",
                                       "flightType": "firstClass", "agency": "Rainbow"})
    print("price  : R$", p["predicted_price"], "| distance", p["derived"]["distance_km"], "km")
    g = call("/predict/gender", {"name": "Charlotte Johnson"})
    print("gender :", g["predicted_gender"], f"(confidence {g['confidence']})")
    h = call("/recommend/hotels", {"destination": "Salvador (BH)", "top_k": 2})
    print("hotels :", [r["hotel"] for r in h["recommendations"]])
else:
    print("SKIPPED - endpoint checks require the running container")

SKIPPED - endpoint checks require the running container


### Latency from inside the container

In [10]:
if DOCKER_OK:
    import urllib.request, statistics
    body = json.dumps({"from": "Sao Paulo (SP)", "to": "Rio de Janeiro (RJ)",
                       "flightType": "firstClass", "agency": "Rainbow"}).encode()
    times = []
    for _ in range(30):
        req = urllib.request.Request("http://localhost:5001/predict/flight-price",
                                     data=body, headers={"Content-Type": "application/json"})
        t0 = time.perf_counter()
        urllib.request.urlopen(req, timeout=10).read()
        times.append((time.perf_counter() - t0) * 1000)
    times.sort()
    print(f"p50 {statistics.median(times):.1f} ms | "
          f"p95 {times[int(0.95*len(times))]:.1f} ms | max {times[-1]:.1f} ms")
    print("PRD budget: 500 ms (includes HTTP round-trip over the port mapping)")
else:
    print("SKIPPED")

SKIPPED


In [11]:
if DOCKER_OK:
    rc, out = sh("docker inspect --format "
                 "\"{{.State.Health.Status}}\" voyage-api-test")
    print("healthcheck status:", out)
    sh("docker rm -f voyage-api-test")
    print("cleaned up test container")
else:
    print("SKIPPED")

SKIPPED


## 6. Everyday commands

```bash
# build
docker build -t voyage-analytics-api .

# run
docker run -d --name voyage-api -p 5000:5000 voyage-analytics-api
curl http://localhost:5000/health

# compose (API only)
docker compose up --build -d
docker compose logs -f api
docker compose down

# compose with the MLflow UI on :5001
docker compose --profile tracking up -d
```

**Rebuilding after retraining.** The models are baked into the image, so a new model means a new
image. That is deliberate — it makes the deployed artifact fully reproducible, and the image tag
becomes the version of record for *both* code and weights:

```bash
python main.py                                   # retrain
docker build -t voyage-analytics-api:$(date +%Y%m%d) .
```

---

## 7. Summary

**Built:** a hardened, multi-stage image serving the three validated models, plus a Compose stack with
an optional MLflow UI.

**Design choices that came from measurements taken earlier in this project**
- 40-second health-check start period, because the cold model load measures ~9 s.
- Exact version pins for scikit-learn / numpy / scipy, because joblib artifacts carry no version
  metadata and a mismatch fails confusingly.
- `save=False` on the reference-data fallback, because the container filesystem is read-only.

**Deliberately excluded:** the training stack, `data/`, notebooks, tests and the hotel-attach model —
the image serves only what passed validation.

---
*Next: Kubernetes (objective #4).*
